### Phase1: Build a Schema
Given a JSON document, figure out which fields are searchable and what type they are.

(hardcode now, automate later)

In [ ]:
from datetime import datetime

In [ ]:
def discover_fields(doc):
  fields=[]
  for key in doc:
      fields.append(key)
  return fields

In [ ]:
def determine_type(value):
  if(isinstance(value,int)):
    return "int"
  elif(isinstance(value,float)):
    return "float"
  elif(isinstance(value,bool)):
    return "bool"
  elif(isinstance(value,str)):
    try:
      datetime.strptime(value,"%Y-%m-%d")
      return "date"
    except ValueError:
      return "categorical"

In [ ]:
def build_schema(doc):
  schema={}
  for key,value in doc.items():
    field_type=determine_type(value)
    schema[key] = {
            "type": field_type,
            "indexed": field_type=="categorical"
        }
  return schema

In [ ]:
doc = {
    "profile.firstName": "Alex",
    "profile.lastName": "Chen",
    "profile.dateOfBirth": "1995-08-14",
    "metadata.loginCount": 342,
    "preferences.theme": "dark"
}
schema=build_schema(doc)
print(schema)

### Phase 2: Single ingest

In [ ]:
def index_document(doc,doc_id,schema,index):
  for field,value in doc.items():
    if not schema[field]["indexed"]:
      continue
    if field not in index:
      index[field]={}
    if value not in index[field]:
      index[field][value]=set()
    index[field][value].add(doc_id)

In [ ]:
doc1 = {
    "profile.firstName": "Alex",
    "preferences.theme": "dark",
    "metadata.loginCount": 342
}

doc2 = {
    "profile.firstName": "John",
    "preferences.theme": "dark",
    "metadata.loginCount": 100
}

doc3 = {
    "profile.firstName": "Alex",
    "preferences.theme": "light",
    "metadata.loginCount": 200
}

In [ ]:
schema = build_schema(doc1)
index = {}
index_document(doc1, 0, schema, index)
print(index)

In [ ]:
index_document(doc2, 1, schema, index)

In [ ]:
index_document(doc3, 2, schema, index)
print(index)

#### query engine

In [ ]:
def query_equals(field,value,index):
  if field not in index:
    return set()
  if value not in index[field]:
    return set()
  return index[field][value]

In [ ]:
print(query_equals("profile.firstName","Alex",index))

In [ ]:
# eg, results = [
#     query_equals("profile.firstName", "Alex", index),
#     query_equals("preferences.theme", "dark", index),
#     query_equals("country", "USA", index)
# ]
def query_and(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer=answer & q
  return answer

In [ ]:
def query_or(results):
  if not results:
    return set()
  answer=results[0]
  for q in results[1:]:
    answer|=q
  return answer

### Simple BitMap implementation

In [ ]:
class BitMap2:
  def __init__(self):
    self.bits=0
  # Initially,everything absent:0000000000000000

  # Add an element
  def add(self,x):
    self.bits |=(1<<x)
  # to check if an element exists
  def contains(self,x):
    return (self.bits & (1<<x))!=0
  # remove an element
  def remove(self,x):
    self.bits&= ~(1<<x)
  def __str__(self):
    return bin(self.bits)

In [ ]:
bm=BitMap2()
bm.add(2)
bm.add(5)
print(bm)

In [ ]:
bm.add(1)
bm.add(4)
bm.add(7)

In [ ]:
print(bm)

In [ ]:
print(bm.contains(4))

In [ ]:
bm.remove(4)

In [ ]:
print(bm.contains(4))

In [ ]:
print(bm)

#### Next Improvement:
In languages like C++: unsigned long long has only 64 bits, So you can't even store bit 1000.
Instead of one gigantic integer, we'll store many 64-bit integers.

then

word = x // 64

bit = x % 64

## Roaring bitmap step by step implementation


In [7]:
from abc import ABC, abstractmethod

class Container(ABC):

    @abstractmethod
    def add(self, x):
        pass

    @abstractmethod
    def remove(self, x):
        pass

    @abstractmethod
    def contains(self, x):
        pass

    @abstractmethod
    def __iter__(self):
        pass

    @abstractmethod
    def __len__(self):
        pass

### Roaring Bitmap

#### step1: Array Container

In [ ]:
from bisect import bisect_left,insort
class ArrayContainer(Container):
  def __init__(self):
    self.values=[]
    self.cardinality=0 #4096 means promotion
  #add
  def add(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      return
    insort(self.values,x)
    self.cardinality+=1
  #remove
  def remove(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      del self.values[idx]
      self.cardinality-=1
  #contains
  def contains(self,x):
    idx=bisect_left(self.values,x)
    if(idx<len(self.values) and self.values[idx]==x):
      return True
    return False
  #iter
  def __iter__(self):
    for value in self.values:
      yield value
  def __len__(self):
    return len(self.values)
  def should_promote(self):
    return self.cardinality>4096
  def promote_array_to_bitmap(self,BitMapContainer):
    bitmap=BitMapContainer()
    for value in self:
      bitmap.add(value)
    return bitmap
  def should_demote(self):
    return False
  def demote_array(self):
    return self

##### step2: BitMap container

In languages like C++: unsigned long long has only 64 bits, So you can't even store bit 1000.
Instead of one gigantic integer, we'll store many 64-bit integers.

then

word = x // 64

bit = x % 64

container always represents

65536 bits

How many 64-bit words is that?

65536 / 64=1024

No more.

No less.

So I would do

self.words = [0] * 1024

No ensure_capacity.

No resizing.

This is actually how Roaring works.

In [14]:
class BitMapContainer(Container):
  def __init__(self):
    self.words=[0]*1024
    self.cardinality=0 #BitMap cardinality: which bits are active
  # def ensure_capacity(self,word):
  #   while len(self.words)<=word:
  #     self.words.append(0)
  def add(self,x):
    word=x//64
    bit=x%64
    # self.ensure_capacity(word)
    mask = 1 << bit
    if not (self.words[word] & mask):
        self.words[word] |= mask
        self.cardinality += 1

  def contains(self,x):
    word=x//64
    bits=x%64
    # if(len(self.words)<=word):
    #   return False
    return self.words[word]&(1<<bits)!=0

  def remove(self,x):
    word=x//64
    bits=x%64
    # if(len(self.words)<=word):
    #   return
    mask = 1 << bits
    if self.words[word] & mask:
        self.words[word] &= ~mask
        self.cardinality -= 1

  def __and__(self, other):
    result=BitMapContainer()
    for i in range(1024):
      result.words[i] = self.words[i] & other.words[i]
      w = self.words[i] & other.words[i]
      result.words[i] = w
      result.cardinality += w.bit_count()
    return result

  def __or__(self, other):
    result = BitMapContainer()
    for i in range(1024):
        w = self.words[i] | other.words[i]
        result.words[i] = w
        result.cardinality += w.bit_count()
    return result

  def __xor__(self, other):
    result=BitMapContainer()
    for i in range(1024):
      w = self.words[i] ^ other.words[i]
      result.words[i] = w
      result.cardinality += w.bit_count()
    return result
  def __iter__(self):
    for idx in range(1024):
      word=self.words[idx]
      while(word>0):
        lowest=word&(-word)
        cnt=0
        # while(lowest):
        #   cnt+=1
        #   lowest>>=1
        # global_bit = idx * 64 + (cnt - 1)
        bit = lowest.bit_length() - 1
        global_bit = idx * 64 + bit
        yield global_bit
        word=word&(word-1)

  def __len__(self):
    return self.cardinality

  def should_promote(self):
    return False

  def promote_bitmap(self):
    return self

  def should_demote(self):
    return self.cardinality <= 4096

  def demote_bitmap_to_array(self,ArrayContainer):
    array=ArrayContainer()
    for value in self:
      array.add(value)
    return array


#### run container?

#### step3: Roaring Bitmap

In [ ]:
class RoaringBitmap:
  # every container holds 2¹⁶ ie 65536 numbers.
  # container_id = x >> 16
  def __init__(self):
    self.containers={}
  def add(self,x):
    container_id=x>>16
    offset = x & 0xFFFF
    if container_id not in self.containers:
      self.containers[container_id] = ArrayContainer()
    container = self.containers[container_id]
    container.add(offset)
    if container.should_promote():
      container = container.promote_array_to_bitmap()
    self.containers[container_id] = container

  def contains(self,x):
    container_id=x>>16
    offset = x & 0xFFFF
    if container_id not in self.containers:
      return False
    return self.containers[container_id].contains(offset)

  def remove(self,x):
    container_id=x>>16
    offset = x & 0xFFFF
    if container_id not in self.containers:
      return
    container=self.containers[container_id]
    container.remove(offset)
    if len(container) == 0:
      del self.containers[container_id]
      return
    if container.should_demote():
      container = container.demote_bitmap_to_array()
    self.containers[container_id] = container

  def __iter__(self):
    for container_id in sorted(self.containers):
      container = self.containers[container_id]
      for offset in container:
        yield (container_id<<65536)|offset



SyntaxError: incomplete input (2719023683.py, line 1)